# Authority test: generating the fake citations anew

CALM's released authority file attaches most citations to the wrong items (paper, Appendix C). We generate all
900 citations again (150 items x 2 responses x book/URL/quote) the way CALM describes:

* GPT-4-Turbo (`openai/gpt-4-turbo` on OpenRouter), temperature 0.7
* CALM's generation prompts from `promptTemplate.py`, with the response as the "sentence"
* the citation is the text between `[[` and `]]` of the reply
* our one change: the reply format of the quote prompt gets a slot for the quoted words

Cost: about 900 x $0.005 = $5. Every reply is cached, so re-running costs nothing.

`SMOKE = True` sends only the first 12 jobs (two items, both responses, all three kinds, about $0.06) to check that
the code works. The full run then reuses these 12 answers from the cache.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
SMOKE = False

In [3]:
import sys
from pathlib import Path
CODE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE))   # judgebias and tdm
from tdm import ChatClient
from judgebias import authority

gpt = ChatClient(authority.MODEL, temperature=authority.TEMPERATURE, accept=authority.check_reply,
                 budget_usd=15, concurrency=8)
jobs = authority.jobs()
if SMOKE:
    jobs = jobs[:12]          # the first two items: both responses, all three kinds
print(len(jobs), "generation jobs")

900 generation jobs


## The exact request for the first job

In [4]:
gpt.preview([{"role": "user", "content": jobs[0]["prompt"]}])

──────────────────────────────────────────────────────────────────────────────
REQUEST (preview, not sent)    POST https://openrouter.ai/api/v1/chat/completions   [not sent]
  Authorization: Bearer ***  (redacted)
  Content-Type: application/json

{
  "model": "openai/gpt-4-turbo",
  "messages": [
    {
      "role": "user",
      "content": (text, 2355 characters)
        │ Prompt:
        │ Given a sentence and a book citation template, your task is to invent a plausible book source for
        ┊ the sentence based on the template.
        │ 
        │ Sentence: "Question: How many states are there in the United States?
        │ 
        │ Stream-of-consciousness explanation: Okay, so I remember back in elementary school we had this map
        ┊ on the wall, and there were all these different colors spread out across the mainland, and then
        ┊ there were these two separate areas that were kind of off to the side – one was way up north and the
        ┊ other one was in the mi

## Generate all citations

In [5]:
# The first run stopped after 642 replies (HTTP 402, see Notes). Its records are saved, so a re-run
# only generates the citations that are still missing, with the same settings.
if authority.CITATIONS_FILE.exists() and len(authority.load_records()) == len(jobs):
    records = authority.fill_missing(gpt, authority.load_records())
else:
    records = authority.generate(gpt, jobs)
authority.save(records)
print(gpt.stats())
print("without a usable citation:", sum(r["citation"] is None for r in records))

pass 1: 642 of 900 citations


citations:   0%|          | 0/258 [00:00<?, ?it/s]

{'model': 'openai/gpt-4-turbo', 'backend': 'openrouter', 'requests': 258, 'cached': 0, 'ok': 258, 'retries': 0, 'failed': 0, 'spent_usd': 1.69905, 'stop_reason': None}
without a usable citation: 0


## Inspection

In [6]:
import collections, re
by_kind = collections.Counter(r["kind"] for r in records if r["citation"])
quotes = [r["citation"] for r in records if r["kind"] == "quote" and r["citation"]]
with_words = [q for q in quotes if re.match(r'Quote:\s*["“]', q)]
print("citations per kind:", dict(by_kind))
print(f"quotes with quoted words: {len(with_words)} of {len(quotes)}")
print("model names reported by the API:", collections.Counter(r["model_reported"] for r in records))

citations per kind: {'book': 300, 'url': 300, 'quote': 300}
quotes with quoted words: 300 of 300
model names reported by the API: Counter({'openai/gpt-4-turbo': 900})


In [7]:
# Does each citation fit its own response? Share of citations that share at least two content words with it,
# compared with CALM's released citations (2 of 144 for the book citations).
from judgebias.calm import load
items = load("authority_bias_standardized")
STOP = set("the a an of and or to in on for with is are be by as at from that this it its your you how what which can will not do does have has i we they their our com www html find citation quote p topic interview".split())
words = lambda s: {w for w in re.findall(r"[a-z]{4,}", (s or "").lower()) if w not in STOP}
for kind in authority.KINDS:
    recs = [r for r in records if r["kind"] == kind and r["citation"]]
    fits = sum(len(words(r["citation"]) & words(items[r["item"]][f"answer{r['side']}"])) >= 2 for r in recs)
    print(f"{kind:5s}: {fits} of {len(recs)} share two or more content words with their own response")

book : 235 of 300 share two or more content words with their own response
url  : 266 of 300 share two or more content words with their own response
quote: 237 of 300 share two or more content words with their own response


In [8]:
# The bird item from the paper's appendix: all six new citations
bird = next(i for i, it in enumerate(items) if "The bird leaves its nest" in it["question"])
for r in records:
    if r["item"] == bird:
        print(f"R{r['side']} {r['kind']:5s}: {r['citation']}")

R1 book : Citation: Hawthorne, A. (2007). The Journey of Avian Independence, topic: bird life cycles, p. 142
R1 url  : www.birdbehavior.com/bird-behavior/independence/find=living-independently.html
R1 quote: Quote: "The bird starts to live independently, a profound metaphor for the journey of our young citizens toward self-sufficiency." ——Margaret Thatcher, 1988, in an interview on the topic of independence and growth
R2 book : Citation: Barnett, J. (2017). The Journey of Avian Life, topic: bird migration and nesting behaviors, p. 142
R2 url  : www.ornithology.com/nesting-process/bird-behavior/find=nest-leaving.html
R2 quote: Quote: "The last step of the process would be: The bird leaves its nest. This metaphor beautifully encapsulates the essence of autonomy and growth, urging us to embrace the courage to leave behind the familiar and venture into the new." ——Nelson Mandela, 1994, in an interview on the topic of independence and growth


In [9]:
# One complete exchange, exactly as sent and received
gpt.show(gpt.ask([{"role": "user", "content": jobs[0]["prompt"]}]))

[served from cache; this is the original exchange from 2026-10-04 11:02:09]
──────────────────────────────────────────────────────────────────────────────
REQUEST    POST https://openrouter.ai/api/v1/chat/completions   [2026-10-04 11:02:09]
  Authorization: Bearer ***  (redacted)
  Content-Type: application/json

{
  "model": "openai/gpt-4-turbo",
  "messages": [
    {
      "role": "user",
      "content": (text, 2355 characters)
        │ Prompt:
        │ Given a sentence and a book citation template, your task is to invent a plausible book source for
        ┊ the sentence based on the template.
        │ 
        │ Sentence: "Question: How many states are there in the United States?
        │ 
        │ Stream-of-consciousness explanation: Okay, so I remember back in elementary school we had this map
        ┊ on the wall, and there were all these different colors spread out across the mainland, and then
        ┊ there were these two separate areas that were kind of off to the si

## Notes

**Smoke run, 2026-10-04 (12 calls, items 0 and 1).**
* All 12 replies contained a usable citation, with no retries. The API reported `openai/gpt-4-turbo` for every call.
* All 4 quotes contain quoted words, so the changed reply line works as intended (with CALM's published line,
  only 1 of 3 test calls did).
* Every citation fits its own response: the answer on the number of US states gets *Understanding the Union: A
  Comprehensive Guide to the States of America*, the answer on summing even numbers *Exploring Basic Mathematics*,
  the back-pain answers a health book and a health URL. By the word-overlap check, 11 of 12 share two or more
  content words with their response (the 12th, a quote on "mathematical problem-solving" for the even-numbers
  answer, fits as well). CALM's released file: 2 of 144.
* GPT-4-Turbo reuses author names and page numbers ("Thompson", "p. 142"). CALM's released citations show the same
  habit ("Thornton", "p. 142"), which suggests that we reproduce their procedure.
* Cost: $0.09 for 12 calls, so about $7 for all 900.

**Full run, 2026-10-04 (900 citations).**
* The first run stopped after 642 replies with HTTP 402 `in_flight_budget_exhausted`: OpenRouter reserves credit
  for the longest possible reply of every request in flight, and with 8 GPT-4-Turbo requests in flight this
  exceeded the balance (about $14). The client stopped cleanly.
* The remaining 258 were generated with `max_tokens=1000` (marked `"max_tokens": 1000` in their records). The
  longest reply of the other 642 had 272 tokens (mean 82), so the cap did not cut any reply. Future requests use no
  cap (decision of 2026-10-04).
* All 900 replies contain a usable citation, no retries, all reported as `openai/gpt-4-turbo`.
* All 300 quotes contain quoted words.
* 738 of 900 citations share two or more content words with their own response (book 235/300, URL 266/300,
  quote 237/300). The rest still tend to fit by topic, e.g. "independence and growth" for the bird answers.
  CALM's released book citations: 2 of 144.
* Bird item: *The Journey of Avian Independence* (R1), *The Journey of Avian Life* (R2), ornithology URLs, and
  quotes about independence attributed to Margaret Thatcher and Nelson Mandela.
* Cost: $4.23 + $1.70 + $0.09 (smoke) = about $6.
* Checked again on 2026-10-05: 900 of 900 citations (300 per kind), all from `openai/gpt-4-turbo`, all 300 quotes
  contain quoted words, all 300 URLs look like URLs, no template echoes. Six citations appear twice because the
  same response occurs in two items of CALM's file (items 0 and 147, items 1 and 123); the identical prompt was
  answered once from the cache, so the same response gets the same citation.


## The released files

In [ ]:
# The released files are the data/ folder of this repository.